# A resource can own its connection without a pool

The direct form calls asyncpg.connect in open and the connection’s close in cleanup. Replace only connect so the example needs no database.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio

from aoa.action_machine.exceptions import HandleError, TransactionLeftOpenError
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection
from aoa.action_machine.resources.postgres import PostgresResource

## 2. Define the objects used in this case

The recording connection logs commands and rejects overlapping operations. The pool limits borrowers; neither class parses SQL or stores database rows.

In [2]:
class RecordingConnection:
    """Record commands; do not parse SQL or simulate database storage."""
    def __init__(self, number):
        """Give this connection a visible identity."""
        self.number = number
        self.commands = []
        self.busy = False
        self.closed = False
    async def execute(self, query, *params):
        """Reject concurrent use and record one command."""
        if self.busy or self.closed:
            raise RuntimeError("connection is busy or closed")
        self.busy = True
        try:
            await asyncio.sleep(0)
            self.commands.append(query)
            return "OK"
        finally:
            self.busy = False
    async def close(self):
        """Mark the physical connection closed."""
        self.closed = True

class RecordingPool:
    """Offer a fixed number of observable connections without a server."""
    def __init__(self, size=1):
        """Create the fixed capacity used by this example."""
        self.connections = [RecordingConnection(i + 1) for i in range(size)]
        self.free = asyncio.Queue()
        for connection in self.connections:
            self.free.put_nowait(connection)
        self.held = set()
        self.closed = False
    async def acquire(self, *, timeout=None):
        """Wait for an available connection with the supplied bound."""
        connection = await asyncio.wait_for(self.free.get(), timeout)
        self.held.add(connection)
        return connection
    async def release(self, connection):
        """Return a borrowed connection exactly once."""
        self.held.remove(connection)
        self.free.put_nowait(connection)
    async def close(self):
        """Close the pool only after all borrowers have returned."""
        assert not self.held
        for connection in self.connections:
            await connection.close()
        self.closed = True

from unittest.mock import AsyncMock, patch

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [3]:
connection = RecordingConnection(1)
connect = AsyncMock(return_value=connection)

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
with patch("aoa.action_machine.resources.postgres.postgres_resource.asyncpg.connect", connect):
    db = PostgresResource({"dsn": "postgresql://example.invalid/demo"})
    await db.open()
    await db.close()
connect.assert_awaited_once_with(dsn="postgresql://example.invalid/demo")
assert connection.closed
print("connect calls:", connect.await_count)
print("physical connection closed:", connection.closed)

connect calls: 1
physical connection closed: True


## 5. Explain the result

No network call occurred in this teaching example. In an application, supplying real connection parameters opens a dedicated connection for that resource. Cleanup closes it instead of returning it to a pool.